# kernel-lab on Colab

Run this from VS Code or Cursor with the Colab extension (Select Kernel → Colab → New Colab Server → T4 GPU), or in the browser (Runtime → Change runtime type → T4 GPU).

The kernel runs on Google's machine, not yours, so it can't see your local files. It works from a clone of your GitHub repo:

1. Set `REPO` below to your repo. For a private repo, setup asks for a fine-grained GitHub token with read access (Contents: read-only). In the browser you can store it as a Colab secret named `GITHUB_TOKEN` instead.
2. Run the setup cells once per runtime.
3. Edit locally, `uv run poe push`, then run **Test and benchmark**. It pulls first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "github.com/<you>/kernel-lab"  # <- change this

import os, subprocess
if not os.path.exists("/content/kernel-lab"):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Colab secrets are browser-only; the editor's Colab kernel can't read them
    if not token:
        from getpass import getpass
        token = getpass("GitHub token with read access (blank for a public repo): ").strip()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    r = subprocess.run(["git", "clone", "-q", url, "/content/kernel-lab"], capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode:
        raise RuntimeError("git clone failed: " + (r.stderr.replace(token, "***") if token else r.stderr))
%cd /content/kernel-lab
!pip -q install pytest
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__, "|", torch.cuda.get_device_name())

## Test and benchmark
Change `EX` to the exercise you're on. Pulls your latest push before testing.
With `AUTO_DISCONNECT = True`, the runtime shuts down after the benchmark so it stops using compute units. Set it to `False` to keep going (for example, to profile).

In [ ]:
EX = "ex01_vector_add"
AUTO_DISCONNECT = True  # shut the runtime down after the benchmark
%cd /content/kernel-lab
!git pull -q && git log --oneline -1
!python -m pytest -q {EX}

In [ ]:
!python -m {EX}.bench
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()  # stops compute-unit billing; reconnect with New Colab Server

## Profile (optional, from exercise 4 on)
Nsight Compute is preinstalled on some runtimes. This profiles every kernel launched by the benchmark; `--set full` is slower but shows memory and warp-stall detail.

In [ ]:
!which ncu && ncu --set basic -k regex:".*_kernel" -c 5 python -m {EX}.bench 2>&1 | tail -60 || echo "ncu not available on this runtime"